# Nearest-neighbor and bilinear interpolation of raw dMRI

This notebook operates directly on diffusion-weighted images. It does not use tensor fields.

## 1. Prepare the Colab environment

In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile

PROJECT_ROOT = Path('/content/dMRI_Interpolation_Repository')
DATA_ROOT = Path('/content/dMRI_Data')
if not PROJECT_ROOT.is_dir():
    from google.colab import files
    print('Upload dMRI_Interpolation_Repository.zip')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if not archives:
        raise FileNotFoundError('No ZIP archive was uploaded.')
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall('/content')
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f'Project directory not found: {PROJECT_ROOT}')
if not (DATA_ROOT / 'TENSOR_2x2x2.nii.gz').is_file():
    from google.colab import files
    print('Upload dMRI_Data.zip')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if not archives:
        raise FileNotFoundError('No dMRI data ZIP archive was uploaded.')
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall('/content')
if not (DATA_ROOT / 'TENSOR_2x2x2.nii.gz').is_file():
    raise FileNotFoundError(f'dMRI data folder not found: {DATA_ROOT}')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(PROJECT_ROOT / 'requirements_colab.txt')
])
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project ready:', PROJECT_ROOT)
print('External dMRI data ready:', DATA_ROOT)

## 2. Load the dMRI study

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from dmri_io import load_dmri_study
from dmri_interpolation import reconstruct_dmri

study = load_dmri_study(PROJECT_ROOT)
SLICE_INDEX = 19
GRADIENT_INDEX = 15
result = reconstruct_dmri(
    study.data,
    factor=2,
    slice_indices=[SLICE_INDEX],
    gradient_indices=[GRADIENT_INDEX],
)
print('dMRI shape:', study.data.shape)

## 3. Compare reference, downsampled image, and reconstructions

In [ ]:
reference = result['reference'][:, :, 0, 0]
low = result['low_resolution'][:, :, 0, 0]
nearest = result['nearest'][:, :, 0, 0]
bilinear = result['bilinear'][:, :, 0, 0]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, image, title in zip(
    axes,
    (reference, low, nearest, bilinear),
    ('Reference', 'Low resolution', 'Nearest neighbor', 'Bilinear'),
):
    ax.imshow(image.T, cmap='gray', origin='lower')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
for name, image in [('nearest', nearest), ('bilinear', bilinear)]:
    rmse = np.sqrt(np.mean((image - reference) ** 2))
    print(f'{name} RMSE: {rmse:.6g}')